# Fraud Detection: Feature Engineering v1

Issue 11. We turn the invoice table (about 4.5 million bills) into **one row per client**, add the client features and prepare the columns for the models.

**Definition of done:** one row per client, no id or target in the features, same columns for train and test.

We follow the workflow from `1.1_intro_to_fe`:
1. Split as early as possible, learned steps are fit on the **training split only**.
2. Fixed steps (aggregating the bills of one client) learn nothing from other clients, so one function `build_features()` in [features.py](features.py) does them for train, validation and test.
3. The test data gets the same treatment: fitted steps are only applied with `transform`, never refitted.

All team decisions for this notebook are in [protocol.md](protocol.md) (topic "Task #11"), the reasons for the features are in [eda_summary.md](eda_summary.md).

In [1]:
# reload features.py when it changes, so a running kernel does not use an old version
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

# helpers from features.py: the fixed steps (build_features) and the rare tariff grouping
from features import build_features, categories_to_keep, group_rare

# show all columns in previews; folders and the random seed used in the whole notebook
pd.set_option("display.max_columns", 40)
CLEAN_DIR = Path("data/clean")
FEATURE_DIR = Path("data/features")
RANDOM_STATE = 42

## 1. Load the cleaned data

The input is the cleaned data from [05_eda.ipynb](05_eda.ipynb) (rules in [cleaning.py](cleaning.py)), not the raw Zindi files. Train and test were cleaned in the same way.

In [2]:
# load the cleaned tables from 05_eda (not the raw Zindi files), with the dates parsed
client_train = pd.read_csv(CLEAN_DIR / "client_train_clean.csv", parse_dates=["creation_date"])
invoice_train = pd.read_csv(CLEAN_DIR / "invoice_train_clean.csv", parse_dates=["invoice_date"], low_memory=False)
client_test = pd.read_csv(CLEAN_DIR / "client_test_clean.csv", parse_dates=["creation_date"])
invoice_test = pd.read_csv(CLEAN_DIR / "invoice_test_clean.csv", parse_dates=["invoice_date"], low_memory=False)

# quick size check of all four tables
print("train:", client_train.shape, invoice_train.shape)
print("test: ", client_test.shape, invoice_test.shape)

train: (135493, 6) (4476749, 16)
test:  (58069, 5) (1939730, 16)


## 2. Selected features and target

These are the features from [eda_summary.md](eda_summary.md), combined with the aggregations asked for in issue 11 (count, mean, std, min, max) and `median` from the outlier decision in `protocol.md`. The target is `target` (1 = fraud, 5.6% of the clients).

| Feature | Built from | How (per client) | Notes |
|---|---|---|---|
| `client_catg` | client table | as integer category | Strongest signal (category 51: 16.9% fraud) |
| `region` | client table | as integer category | Clear differences between big regions |
| `tarif_type` | invoices | most common plan, rare plans (under 1% of train clients) grouped as `-1` ("other") | Plan 10 is above average on electricity. The "other" group is learned on train only |
| `n_bills` | invoices | number of bills | Fraud rate rises from 1.3% to 9.0% |
| `cons_mean`, `cons_median`, `cons_std`, `cons_min`, `cons_max` | invoices | statistics of the total consumption per bill (sum of the four levels) | Outliers are kept, no capping |
| `consommation_level_k_sum`, `_max` (k = 1 to 4) | invoices | the four original level columns: sum over all bills and the biggest single bill per level | No new level features, no mean (it is the sum divided by `n_bills`) |
| `backward_count`, `has_backward` | invoices | bills with `new_index < old_index`, and whether there is any | Twice the fraud rate |
| `max_change` | invoices | biggest jump in consumption between two bills of the same meter type | Fraud clients are less steady |
| `cons_std_elec`, `cons_std_gaz`, `max_change_elec`, `max_change_gaz` | invoices | same statistics, only on electricity or gas bills. NaN = no bill on this meter | Mainly an electricity signal. Keep only if validation improves |
| **target** | client table (train only) | not a feature, kept in `y` | |

Not used: `client_id` (only the row index, for the join and the submission), `reading_remarque`, `mismatch_share`, meter setup, and time features (open question in `protocol.md`).

Missing values: NaN stays for the tree models. The standard deviation and the biggest jump of a client with one bill are set to 0.

## 3. Train-validation split

We split the **clients** of the Zindi train set (stratified on `target`, 80% train, 20% validation) before we build any feature. One row per client means a client is never in both parts. The Zindi test set has no target: it only gets the features, and we do not score it.

In [3]:
# split the client ids 80/20, stratified on target so both parts keep the same fraud rate
train_ids, valid_ids = train_test_split(
    client_train["client_id"],
    test_size=0.20,
    stratify=client_train["target"],
    random_state=RANDOM_STATE,
)

# a client and all of their bills go into the same part
client_fit = client_train[client_train["client_id"].isin(train_ids)]
client_valid = client_train[client_train["client_id"].isin(valid_ids)]
invoice_fit = invoice_train[invoice_train["client_id"].isin(train_ids)]
invoice_valid = invoice_train[invoice_train["client_id"].isin(valid_ids)]

# check that the fraud rate is about the same in both parts
print(f"train clients: {len(client_fit):,} (fraud {client_fit['target'].mean():.2%})")
print(f"valid clients: {len(client_valid):,} (fraud {client_valid['target'].mean():.2%})")

train clients: 108,394 (fraud 5.58%)
valid clients: 27,099 (fraud 5.58%)


## 4. Fixed steps: aggregate the invoices per client

`build_features()` ([features.py](features.py)) is called in the same way for train, validation and test. It takes the cleaned client and invoice table and returns one row per client. It never sees the target.

In [4]:
# same fixed function for train, validation and test: one row per client, no target
X_train = build_features(client_fit, invoice_fit)
X_valid = build_features(client_valid, invoice_valid)
X_test = build_features(client_test, invoice_test)

# the target stays separate, in the same client order as the features
y_train = client_fit.set_index("client_id")["target"].loc[X_train.index]
y_valid = client_valid.set_index("client_id")["target"].loc[X_valid.index]

# check the shapes and look at the first rows
print(X_train.shape, X_valid.shape, X_test.shape)
X_train.head()

(108394, 24) (27099, 24) (58069, 24)


,n_bills,cons_mean,cons_median,cons_std,cons_min,cons_max,consommation_level_1_sum,consommation_level_1_max,consommation_level_2_sum,consommation_level_2_max,consommation_level_3_sum,consommation_level_3_max,consommation_level_4_sum,consommation_level_4_max,backward_count,has_backward,max_change,cons_std_elec,max_change_elec,cons_std_gaz,max_change_gaz,tarif_type,client_catg,region
client_id,,,,,,,,,,,,,,,,,,,,,,,,
train_Client_0,35,362.971429,267.0,341.553930,38,1386,12334,1200,370,186,0,0,0,0,0,0,1346.0,341.553930,1346.0,NaN,NaN,11,11,101
train_Client_1,37,557.540541,520.0,197.935960,190,1207,20629,1207,0,0,0,0,0,0,0,0,732.0,197.935960,732.0,NaN,NaN,11,11,107
train_Client_100,20,1.200000,0.0,3.607011,0,15,24,15,0,0,0,0,0,0,0,0,15.0,3.607011,15.0,NaN,NaN,11,11,105
train_Client_1000,14,922.642857,770.0,633.485669,124,2382,9292,800,1468,400,1643,800,514,382,0,0,1646.0,633.485669,1646.0,NaN,NaN,11,11,303
train_Client_100000,40,286.575000,177.5,338.205437,0,1462,11201,1200,262,262,0,0,0,0,0,0,1106.0,367.968463,1106.0,139.262824,476.0,11,11,309


### Quick check against the EDA

The new features should show the same signal as in [eda_summary.md](eda_summary.md): category 51 and backward steps have a higher fraud rate, and more bills mean more fraud. (Numbers differ a little because this is only the 80% training split.)

In [5]:
# add the target back only for this check, to compare the fraud rates with the EDA
check = X_train.assign(target=y_train)
print("fraud rate overall:", f"{check['target'].mean():.1%}")
print("client_catg 51:    ", f"{check.loc[check['client_catg'] == 51, 'target'].mean():.1%}")
print("has_backward = 1:  ", f"{check.loc[check['has_backward'] == 1, 'target'].mean():.1%}", f"({(check['has_backward'] == 1).sum()} clients)")
print()
# fraud rate and number of clients by number of bills
print(check.groupby(pd.cut(check["n_bills"], [0, 10, 50, np.inf], labels=["1-10", "11-50", "51+"]), observed=True)["target"].agg(["mean", "size"]))

fraud rate overall: 5.6%
client_catg 51:     16.9%
has_backward = 1:   11.1% (1443 clients)

             mean   size
n_bills                 
1-10     0.013008  26983
11-50    0.061189  54650
51+      0.088113  26761


## 5. Steps that learn from the data (fit on train only)

Only the training split may teach us something. Two things are learned:
- Which tariff plans are common enough to keep (at least 1% of the training clients). The others become `-1` ("other"). Plan 40 is kept: it covers about 7% of the training clients (gas bills only, see `protocol.md`). Only plans under 1% are grouped, because such small groups can look extreme by chance.
- The scaling and encoding of the logistic regression baseline (section 6).

In [6]:
# learn on train only which plans cover at least 1% of the training clients
keep_plans = categories_to_keep(X_train["tarif_type"], min_share=0.01)
print("tariff plans kept:", sorted(keep_plans))

# apply the same list to all three tables, so other plans become -1 ("other") everywhere
for table in (X_train, X_valid, X_test):
    table["tarif_type"] = group_rare(table["tarif_type"], keep_plans)

# check how many training clients are in each plan now
X_train["tarif_type"].value_counts()

tariff plans kept: [10, 11, 40]


tarif_type
 11    91269
 10     9058
 40     7263
-1       804
Name: count, dtype: int64

**Tree models (LightGBM, XGBoost):** the table is ready. `client_catg`, `region` and `tarif_type` are integer categories, NaN stays as it is.

## 6. Version for the logistic regression baseline

The baseline needs numbers on a similar scale and no codes that look like sizes:
- NaN (clients without a gas or electricity bill) becomes 0, and a column marks where it happened. The fill value is fixed, the columns to mark are found on train.
- `log1p` for all counts and consumption values (skewed, see the log transform in `protocol.md`), then standard scaling fitted on train.
- One-hot encoding for `client_catg`, `region` and `tarif_type`. Categories that are not in train are ignored.
- `has_backward` stays 0 or 1.

In [7]:
# sort the columns into categories, the 0/1 flag and numeric columns
category_columns = ["client_catg", "region", "tarif_type"]
flag_columns = ["has_backward"]
numeric_columns = [c for c in X_train.columns if c not in category_columns + flag_columns]

# log1p needs values of 0 or more, so check this before building the steps
assert (X_train[numeric_columns].min() >= 0).all(), "log1p needs values of 0 or more"

# numeric columns: NaN becomes 0 plus a missing marker, then log1p, then standard scaling
numeric_steps = Pipeline([
    ("fill", SimpleImputer(strategy="constant", fill_value=0, add_indicator=True)),
    ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("scale", StandardScaler()),
])

# one-hot for the categories (categories not in train are ignored), has_backward passes through as 0/1
logreg_preprocessor = ColumnTransformer(
    [
        ("numeric", numeric_steps, numeric_columns),
        ("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False), category_columns),
    ],
    remainder="passthrough",
    verbose_feature_names_out=False,
).set_output(transform="pandas")

# fit on train only, then transform all three
X_train_logreg = logreg_preprocessor.fit_transform(X_train)
X_valid_logreg = logreg_preprocessor.transform(X_valid)
X_test_logreg = logreg_preprocessor.transform(X_test)

# check the shapes and show the first rows and columns
print(X_train_logreg.shape, X_valid_logreg.shape, X_test_logreg.shape)
X_train_logreg.iloc[:3, :8]

(108394, 57) (27099, 57) (58069, 57)


,n_bills,cons_mean,cons_median,cons_std,cons_min,cons_max,consommation_level_1_sum,consommation_level_1_max
client_id,,,,,,,,
train_Client_0,0.442474,0.087373,0.188222,0.213820,0.718675,0.181441,0.390123,0.283575
train_Client_1,0.494706,0.376794,0.544283,-0.114606,1.384185,0.098515,0.629674,0.287430
train_Client_100,-0.078220,-3.365128,-2.806433,-2.390187,-0.815986,-2.496569,-2.498300,-2.580617


The output shows only the first 8 columns (scaled numbers). The one-hot columns come after the numeric and missing-marker columns, here they are:

In [8]:
# find the one-hot columns by their prefix (category column name + "_") and show them
one_hot_columns = [c for c in X_train_logreg.columns if c.startswith(tuple(f"{name}_" for name in category_columns))]
print(len(one_hot_columns), "one-hot columns:", one_hot_columns)
X_train_logreg[one_hot_columns].head(3)

32 one-hot columns: ['client_catg_11', 'client_catg_12', 'client_catg_51', 'region_101', 'region_103', 'region_104', 'region_105', 'region_106', 'region_107', 'region_199', 'region_206', 'region_301', 'region_302', 'region_303', 'region_304', 'region_305', 'region_306', 'region_307', 'region_308', 'region_309', 'region_310', 'region_311', 'region_312', 'region_313', 'region_371', 'region_372', 'region_379', 'region_399', 'tarif_type_-1', 'tarif_type_10', 'tarif_type_11', 'tarif_type_40']


,client_catg_11,client_catg_12,client_catg_51,region_101,region_103,region_104,region_105,region_106,region_107,region_199,region_206,region_301,region_302,region_303,region_304,region_305,region_306,region_307,region_308,region_309,region_310,region_311,region_312,region_313,region_371,region_372,region_379,region_399,tarif_type_-1,tarif_type_10,tarif_type_11,tarif_type_40
client_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
train_Client_0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
train_Client_1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
train_Client_100,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


## 7. Definition of done: checks

- One row per client.
- No id or target in the features.
- Same columns, in the same order, for train, validation and test.
- Every Zindi test client has a row (Zindi needs a prediction for each one).

In [9]:
# each table: every client only once, and exactly the clients of that part
for name, table, clients in [("train", X_train, client_fit), ("valid", X_valid, client_valid), ("test", X_test, client_test)]:
    assert table.index.is_unique, f"{name}: duplicate clients"
    assert set(table.index) == set(clients["client_id"]), f"{name}: clients do not match"

# no id or target in the features, same columns in the same order, no client in both train and valid
assert not {"client_id", "target"} & set(X_train.columns)
assert list(X_train.columns) == list(X_valid.columns) == list(X_test.columns)
assert list(X_train_logreg.columns) == list(X_valid_logreg.columns) == list(X_test_logreg.columns)
assert len(set(X_train.index) & set(X_valid.index)) == 0

print("All checks passed.")
print(f"{X_train.shape[1]} tree features, {X_train_logreg.shape[1]} columns for the logistic regression baseline")
print()
# columns that still have NaN (kept on purpose for the tree models)
print("Missing values (tree version, on purpose):")
print(X_train.isna().sum()[lambda s: s > 0])

All checks passed.
24 tree features, 57 columns for the logistic regression baseline

Missing values (tree version, on purpose):
cons_std_elec        554
max_change_elec      554
cons_std_gaz       58567
max_change_gaz     58567
dtype: int64


## 8. Save the tables

The tables are saved in `data/features/` (ignored by git, like all data) so the model notebooks can load them directly. The row index is `client_id`; the target is in separate files.

In [10]:
# create data/features/ if it does not exist yet
FEATURE_DIR.mkdir(exist_ok=True)

# save every table as CSV with client_id as the index, so the model notebooks can load them
for name, table in {
    "X_train": X_train, "X_valid": X_valid, "X_test": X_test,
    "X_train_logreg": X_train_logreg, "X_valid_logreg": X_valid_logreg, "X_test_logreg": X_test_logreg,
    "y_train": y_train, "y_valid": y_valid,
}.items():
    table.to_csv(FEATURE_DIR / f"{name}.csv")

# list the saved files as a check
print(sorted(path.name for path in FEATURE_DIR.iterdir()))

['X_test.csv', 'X_test_logreg.csv', 'X_train.csv', 'X_train_logreg.csv', 'X_valid.csv', 'X_valid_logreg.csv', 'y_train.csv', 'y_valid.csv']


## Summary and next steps

- One row per client for train (80%), validation (20%) and the Zindi test set, with the same columns.
- Fixed steps are in `build_features()`, learned steps (rare tariff plans, scaling, encoding) are fitted on the training split only.
- The baseline (logistic regression on the `_logreg` version) is in [09_baseline-model.ipynb](09_baseline-model.ipynb). The per-meter columns improve the validation score slightly (ROC-AUC 0.7565 vs 0.7531), so we keep them. Next: tree models (LightGBM, XGBoost) on `X_train` / `X_valid`.
- No time features in v1 (team decision, see [protocol.md](protocol.md)).